# Patent phrase matching: controlled experiments

Utkrisht Suman · A24ARIU0001

Implementation prepared for review. Run the notebook to collect the experiment outputs; no training results or human ratings have been assumed.

# Patent phrase matching: starter notebook

This notebook is the **minimum working pipeline** for the mini project described in [project_brief.md](project_brief.md). It fine-tunes a pretrained Transformer **cross-encoder** to predict how similar two patent phrases are, checks it on a validation split, and writes `submission.csv`.

It is a starting point, not a solution. The baseline comparison, the experiments, and the error analysis in the brief are your work.

**Running it**

- Kaggle notebook, **GPU T4 or P100**, attached inputs:
  1. the competition data, via **Add Input → Competitions → U.S. Patent Phrase to Phrase Matching**;
  2. the output of [download_weights.ipynb](download_weights.ipynb), the saved pretrained checkpoint.
- Turn **Internet off** before you submit, because the competition runs submitted notebooks offline. The notebook falls back to the Hugging Face Hub only when it cannot find a local checkpoint, which works only with internet on.
- Writes to `/kaggle/working/`: `submission.csv` and `val_predictions.csv`. A rerun overwrites both. No checkpoints are saved.
- Uses packages preinstalled on Kaggle (`torch`, `transformers`, `datasets`, `scikit-learn`, `pandas`).

**Run package:** execute the full notebook. Written observations mentioning earlier CPU checks are reference observations; send your outputs back so they can be updated for your run.


In [ ]:
"""Prepare the offline patent runtime without replacing compatible image packages."""
from pathlib import Path
import importlib
import json
import subprocess
import sys
from importlib import metadata

ROOTS = [
    "transformers==5.22.2", "accelerate==1.15.0", "datasets==5.1.0",
    "scikit-learn>=1.4", "sentencepiece==0.2.3", "protobuf",
    "pandas>=2.1", "matplotlib>=3.8",
]

if Path("/kaggle/input").exists():
    candidates = sorted(Path("/kaggle/input").rglob("patent_runtime_wheels/wheels_manifest.json"))
    assert candidates, "Attach PATENT_DOWNLOAD_WEIGHTS output, including patent_runtime_wheels."
    manifest_path = candidates[0]
    manifest = json.loads(manifest_path.read_text())
    assert manifest["python"].split(".")[:2] == [str(sys.version_info.major), str(sys.version_info.minor)], \
        "The weights/wheels and notebook need the same Python major/minor version."
    roots = manifest.get("root_requirements", ROOTS)
    torch_before = metadata.version("torch")
    constraints = Path("/kaggle/working/patent_keep_cuda_torch.txt")
    constraints.write_text("torch==" + torch_before + "\n")
    base = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
            "--no-index", "--find-links", str(manifest_path.parent), "-c", str(constraints)]
    report_path = Path("/kaggle/working/patent_runtime_plan.json")
    subprocess.run([*base, "--dry-run", "--report", str(report_path), *roots], check=True)
    planned = json.loads(report_path.read_text()).get("install", [])
    print("Offline package changes:", [p["metadata"]["name"] for p in planned], flush=True)

    # A loaded, unchanged dependency does not require a restart. Check the actual
    # module files belonging to distributions that pip proposes to replace.
    loaded_changes = {}
    loaded_files = {}
    for name, module in list(sys.modules.items()):
        origin = getattr(module, "__dict__", {}).get("__file__") if module else None
        if origin:
            loaded_files[name] = Path(origin).resolve()
    for package in planned:
        name = package["metadata"]["name"]
        try:
            dist = metadata.distribution(name)
        except metadata.PackageNotFoundError:
            continue
        files = {Path(dist.locate_file(f)).resolve() for f in dist.files or []}
        loaded = sorted(name for name, path in loaded_files.items() if path in files)
        if loaded:
            loaded_changes[name] = loaded[:8]
    if loaded_changes:
        raise RuntimeError("A fresh kernel is required BEFORE replacing loaded packages: "
                           + json.dumps(loaded_changes, sort_keys=True))
    if planned:
        subprocess.run([*base, *roots], check=True)
        importlib.invalidate_caches()
    assert metadata.version("torch") == torch_before, "Kaggle CUDA torch must be preserved."
    print("Offline runtime ready; compatible packages and CUDA torch preserved.", flush=True)
else:
    print("Local patent run: install requirements.txt and set PATENT_* paths first.")


## 1. Setup

In [ ]:
import glob
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
transformers.set_seed(SEED)   # seeds Python, NumPy, and PyTorch; GPU results can still vary slightly between runs
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)
import inspect
# Preserve the starter 10% warmup across Transformers API versions.
WARMUP = ({"warmup_ratio": 0.1} if "warmup_ratio" in inspect.signature(TrainingArguments).parameters
          else {"warmup_steps": 0.1})

if Path("/kaggle/working").exists():
    assert torch.cuda.is_available(), "Enable a Kaggle T4 GPU before running the patent training notebook."


## 2. Configuration

These are the settings you will change in your experiments. Change **one** setting at a time so you can tell what caused a difference.

In [ ]:
import json, time, gc
from pathlib import Path
MODEL_ID = "microsoft/deberta-v3-small"
MAX_LEN, EPOCHS, LR, BATCH_SIZE, VAL_FRACTION = 64, 3, 2e-5, 32, 0.2
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("patent_outputs")
OUT.mkdir(parents=True, exist_ok=True)
OUT_DIR = str(OUT)
def find_one(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    return os.path.dirname(hits[0]) if hits else None
DATA_DIR = os.environ.get("PATENT_DATA_DIR") or find_one("us-patent-phrase-to-phrase-matching/train.csv")
assert DATA_DIR, "Attach the competition data or set PATENT_DATA_DIR to its local folder."
MODEL_PATH = os.environ.get("PATENT_MODEL_PATH") or find_one("deberta-v3-small/config.json")
assert MODEL_PATH, "Attach offline pretrained weights or set PATENT_MODEL_PATH; no Hub fallback."
print("data:", DATA_DIR, "| offline model:", MODEL_PATH)

## 3. Data

Each row asks how similar a **target** phrase is to an **anchor** phrase, judged within a patent **context**. The context is a CPC (Cooperative Patent Classification) code such as `G02`, whose first letter is the section. Scores are 0, 0.25, 0.5, 0.75, or 1. The visible `test.csv` has only a few rows; when you submit, Kaggle reruns the notebook with the hidden test set in its place.

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "| test (visible):", test.shape)
print(train.score.value_counts().sort_index())
train.head()

## 4. Model input

The model reads the anchor as the first text segment, and the target plus a short context description as the second. The tokenizer joins them as `[CLS] anchor [SEP] target; context [SEP]`, so attention can compare the two phrases token by token: this is what makes it a cross-encoder. Here the context is just the CPC section name. How you represent context is one of the experiments you can run.

In [ ]:
CPC_SECTIONS = {
    "A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics", "H": "electricity",
}


def add_inputs(df):
    df = df.copy()
    df["text_a"] = df["anchor"]
    df["text_b"] = df["target"] + "; " + df["context"].str[0].map(CPC_SECTIONS)
    return df


train, test = add_inputs(train), add_inputs(test)
train[["text_a", "text_b", "score"]].head()

## 5. Validation split

The training set contains only about 730 distinct anchors, each paired with many targets. If the same anchor appeared in both the training and validation sets, the validation score would overestimate performance on unseen anchors. The split therefore holds out whole **anchors**.

Keep this split fixed (same `SEED` and `VAL_FRACTION`) for every experiment, so your comparisons are fair.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn, val = train.iloc[train_idx].reset_index(drop=True), train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"train rows {len(trn)} | val rows {len(val)} | val anchors {val.anchor.nunique()}")

## 6. Tokenization

Padding is left to the data collator, which pads each batch only to its longest sequence. The printed example should start with `[CLS]` and have a `[SEP]` between the two segments.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# transformers 5.0 builds this tokenizer without [CLS]/[SEP]; set the pair template explicitly
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)])


def to_dataset(df, with_labels=True):
    cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
    ds = Dataset.from_pandas(df[cols].astype({"score": "float32"}) if with_labels else df[cols])  # MSE loss needs float32 labels
    ds = ds.map(lambda b: tokenizer(b["text_a"], b["text_b"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text_a", "text_b"])
    return ds.rename_column("score", "labels") if with_labels else ds


ds_trn, ds_val, ds_test = to_dataset(trn), to_dataset(val), to_dataset(test, with_labels=False)
print(tokenizer.decode(ds_trn[0]["input_ids"]))

## 7. Fine-tuning

With `num_labels=1`, the classification head outputs a single number. Because the labels are floats, Hugging Face treats the task as **regression** and minimizes mean squared error. The competition metric is the **Pearson correlation** between predictions and true scores. Pearson correlation ignores any linear rescaling of the predictions, so the model is rewarded for ranking pairs in the right order, not for hitting the exact values.

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"pearson": float(np.corrcoef(preds.reshape(-1), labels)[0, 1])}


# Load in fp32: the deberta-v3 checkpoints are stored in fp16, and fp16=True mixed precision below needs
# fp32 weights ("Attempting to unscale FP16 gradients" otherwise). Older transformers call this torch_dtype.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=1, dtype=torch.float32)
args = TrainingArguments(
    output_dir=f"{OUT_DIR}/trainer_tmp",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2 * BATCH_SIZE,
    **WARMUP,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",          # keeps disk use and runtime low; the final weights stay in memory
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_trn, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
baseline_start = time.perf_counter()
trainer.train()
baseline_minutes = (time.perf_counter() - baseline_start) / 60
baseline_peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else None

## 8. Validation predictions

The predictions are saved so you can do your error analysis without retraining.

In [ ]:
val["pred"] = trainer.predict(ds_val).predictions.reshape(-1)
val_pearson = np.corrcoef(val["pred"], val["score"])[0, 1]
print(f"Validation Pearson: {val_pearson:.4f}")
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(f"{OUT_DIR}/val_predictions.csv", index=False)

## 9. Submission

The competition expects `/kaggle/working/submission.csv` with columns `id,score` and one row per test row. Clipping to [0, 1] keeps the scores in the label range; it does not change the validation Pearson correlation much.

In [ ]:
test_pred = trainer.predict(ds_test).predictions.reshape(-1)
submission = pd.DataFrame({"id": test["id"], "score": np.clip(test_pred, 0, 1)})
assert len(submission) == len(test) and submission["score"].notna().all()
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
submission.head()

## Baselines and experiments

The first run above reproduces the supplied DeBERTa-v3-small baseline with the original section-name context, seed 42, anchor-grouped split, maximum length 64, batch 32, learning rate 2e-5 and three epochs. Holding out whole anchors prevents the model from seeing the same anchor with different targets during training and validation. Random row splitting would allow that overlap and could overstate generalization to new anchors.

The following no-training baseline encodes each phrase independently with all-MiniLM-L6-v2 and uses cosine similarity. Unlike a cross-encoder it cannot perform joint token-level attention across the pair. The comparison includes both task fine-tuning and checkpoint/architecture differences, so it cannot isolate fine-tuning alone.

The two controlled experiments change only context representation: remove it, or use the raw CPC code. All training settings, validation IDs, tokenizer and checkpoint remain fixed. Test labels are never loaded. Every score below comes from the competition training partition held out for validation.

In [ ]:
baseline_val = val.copy()
baseline_val["pred"] = baseline_val["pred"].astype(float)
records = [{"name": "section_baseline", "changed": "none (starter)",
            "pearson": float(val_pearson), "minutes": baseline_minutes,
            "peak_GB": baseline_peak, "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad)}]
prediction_frames = {"section_baseline": baseline_val}
baseline_val.to_csv(OUT / "val_section_baseline.csv", index=False)
# Release Trainer and optimizer states before another model is loaded.
del trainer, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModel
import torch.nn.functional as F
EMBED_ID = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_PATH = os.environ.get("PATENT_EMBED_PATH") or find_one("all-MiniLM-L6-v2/config.json")
assert EMBED_PATH, "Attach offline MiniLM weights from the download notebook."
embed_tok = AutoTokenizer.from_pretrained(EMBED_PATH, local_files_only=True)
embed_model = AutoModel.from_pretrained(EMBED_PATH, local_files_only=True).to(device).eval()
@torch.no_grad()
def encode_phrases(phrases, batch_size=64):
    chunks = []
    for start in range(0, len(phrases), batch_size):
        batch = embed_tok(phrases[start:start + batch_size], padding=True, truncation=True,
                          max_length=256, return_tensors="pt").to(device)
        hidden = embed_model(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1e-9)
        chunks.append(F.normalize(pooled, p=2, dim=1).cpu())
    return torch.cat(chunks)
def safe_pearson(y, pred):
    y, pred = np.asarray(y, float), np.asarray(pred, float)
    if len(y) < 2 or np.std(y) == 0 or np.std(pred) == 0:
        return float("nan")
    return float(np.corrcoef(y, pred)[0, 1])
start = time.perf_counter()
a = encode_phrases(val.anchor.tolist()); b = encode_phrases(val.target.tolist())
cosine_pred = (a * b).sum(1).numpy()
embed_pearson = safe_pearson(val.score, cosine_pred)
print("No-training MiniLM Pearson:", embed_pearson, "| fine-tuned baseline:", val_pearson)
val.assign(pred=cosine_pred).to_csv(OUT / "val_minilm.csv", index=False)
records.append({"name": "minilm_no_training", "changed": "bi-encoder; different pretrained checkpoint; no task training",
                "pearson": embed_pearson, "minutes": 0., "inference_seconds": time.perf_counter() - start,
                "peak_GB": None, "trainable": 0})
del embed_model, a, b
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
def inputs_for_context(df, mode):
    df = df.copy()
    df["text_a"] = df["anchor"]
    if mode == "section":
        df["text_b"] = df.target + "; " + df.context.str[0].map(CPC_SECTIONS)
    elif mode == "none":
        df["text_b"] = df.target
    elif mode == "code":
        df["text_b"] = df.target + "; " + df.context
    else:
        raise ValueError(mode)
    return df

def run_context_experiment(mode):
    transformers.set_seed(SEED)
    trn_variant = inputs_for_context(trn, mode)
    val_variant = inputs_for_context(val, mode)
    ds_a, ds_b = to_dataset(trn_variant), to_dataset(val_variant)
    fresh = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=1, dtype=torch.float32, local_files_only=True)
    experiment_args = TrainingArguments(output_dir=str(OUT / ("trainer_" + mode)),
        num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=2 * BATCH_SIZE,
        **WARMUP, weight_decay=0.01, eval_strategy="epoch", save_strategy="no",
        logging_steps=100, fp16=torch.cuda.is_available(), report_to="none", seed=SEED)
    experiment_trainer = Trainer(model=fresh, args=experiment_args, train_dataset=ds_a,
        eval_dataset=ds_b, data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter(); experiment_trainer.train()
    minutes = (time.perf_counter() - start) / 60
    peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else None
    frame = val_variant.copy()
    frame["pred"] = experiment_trainer.predict(ds_b).predictions.reshape(-1)
    frame.to_csv(OUT / f"val_context_{mode}.csv", index=False)
    result = {"name": f"context_{mode}", "changed": f"context only: {mode}",
              "pearson": safe_pearson(frame.score, frame.pred), "minutes": minutes,
              "peak_GB": peak, "trainable": sum(p.numel() for p in fresh.parameters() if p.requires_grad)}
    del experiment_trainer, fresh
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result, frame

for mode in ("none", "code"):
    result, frame = run_context_experiment(mode)
    records.append(result); prediction_frames[result["name"]] = frame
    pd.DataFrame(records).to_csv(OUT / "experiments.csv", index=False)
    print(result, flush=True)
experiment_table = pd.DataFrame(records)
base_r = float(records[0]["pearson"])
experiment_table["delta_from_starter"] = experiment_table.pearson - base_r
experiment_table["interpretation"] = [
    "starter baseline" if name == "section_baseline" else
    "architecture and training differ; not a controlled context experiment" if name == "minilm_no_training" else
    "difference <= 0.005; needs another seed" if abs(r - base_r) <= 0.005 else
    "higher on this split; one seed only" if r > base_r else "lower on this split; one seed only"
    for name, r in zip(experiment_table.name, experiment_table.pearson)]
experiment_table.to_csv(OUT / "experiments.csv", index=False)
print(experiment_table.to_string(index=False))
# Only trained cross-encoders are candidates for the final offline retraining notebook.
candidates = [r for r in records if r["name"] in prediction_frames]
best = max(candidates, key=lambda r: r["pearson"])
mode_for_name = {"section_baseline": "section", "context_none": "none", "context_code": "code"}
best_config = {"context_mode": mode_for_name[best["name"]], "selected_by": "validation Pearson",
               "name": best["name"], "validation_pearson": best["pearson"], "seed": SEED,
               "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE, "max_len": MAX_LEN,
               "val_fraction": VAL_FRACTION, "model_id": MODEL_ID}
(OUT / "best_config.json").write_text(json.dumps(best_config, indent=2))
best_val = prediction_frames[best["name"]].copy()
best_val.to_csv(OUT / "val_predictions.csv", index=False)
print("Selected by validation only:", best_config)

In [ ]:
import matplotlib.pyplot as plt
best_val["section"] = best_val.context.str[0]
section_rows = []
for section in CPC_SECTIONS:
    group = best_val[best_val.section == section]
    section_rows.append({"section": section, "n": len(group), "pearson": safe_pearson(group.score, group.pred)})
section_scores = pd.DataFrame(section_rows)
score_means = best_val.groupby("score").agg(n=("pred", "size"), mean_pred=("pred", "mean")).reset_index()
print("Per-section validation Pearson (undefined for constant/undersized groups):")
print(section_scores.to_string(index=False)); print(score_means.to_string(index=False))
section_scores.to_csv(OUT / "section_scores.csv", index=False)
score_means.to_csv(OUT / "score_means.csv", index=False)
plt.figure(figsize=(5, 3.4))
plt.plot(score_means.score, score_means.mean_pred, "o-", label="mean prediction")
plt.plot([0, 1], [0, 1], "--", label="ideal")
plt.xlabel("True score"); plt.ylabel("Mean prediction"); plt.legend(); plt.tight_layout()
plt.savefig(OUT / "score_compression.png", dpi=160); plt.show()
errors = best_val.assign(abs_error=(best_val.pred - best_val.score).abs()).nlargest(10, "abs_error")
error_file = OUT / "top_errors_for_review.csv"
if error_file.exists():
    prior = pd.read_csv(error_file).set_index("id")
    for col in ("failure_type", "evidence", "label_debatable"):
        errors[col] = errors.id.map(prior[col]) if col in prior else ""
else:
    errors["failure_type"] = ""; errors["evidence"] = ""; errors["label_debatable"] = ""
errors.to_csv(error_file, index=False)
print(errors[["id", "anchor", "target", "context", "score", "pred", "abs_error"]].to_string(index=False))
valid_sections = section_scores.dropna(subset=["pearson"]).sort_values("pearson")
print("Lowest measured sections:", valid_sections.head(2).to_dict("records"))
print("Inspect the ten errors and fill 2-3 failure types plus evidence and label-debatability notes in", error_file)
# Save every submission, including failures; scores must be copied from Kaggle, never guessed.
ledger = OUT / "leaderboard_submissions.csv"
if not ledger.exists():
    pd.DataFrame(columns=["version", "notebook_url", "status", "public_score", "private_score",
                          "screenshot_file", "notes"]).to_csv(ledger, index=False)
print("Record actual Kaggle results in", ledger)

## Interpretation and report

The per-section table identifies the hardest sections on this validation sample; a small section can have an unstable correlation. Specialist vocabulary, abbreviations and different label mixes are hypotheses to check against the displayed pairs, not established explanations from the section label alone. Predictions at true scores 0 and 1 can be compared with the diagonal to assess compression toward the more common middle labels. Pearson alone does not reveal calibration.

For the ten largest errors, use the actual anchor, target and CPC context to assign two or three failure categories and write evidence in `top_errors_for_review.csv`. Do not automatically call a label wrong just because the model disagrees. Record every late submission in `leaderboard_submissions.csv`, attach the screenshot, and compare validation/public/private scores. Different anchor and section mixes, sampling noise and selection on validation can produce gaps. Never use private scores to select another model.

Run the companion `patent_report.py` on the output folder after completing those review notes and the submission ledger. It refuses missing evidence rather than creating a final report with invented results. Read and verify the generated four-page PDF before submission.

References: course starter and project brief by bk-anupam; He et al. (2021), DeBERTaV3, arXiv:2111.09543; Reimers & Gurevych (2019), Sentence-BERT, arXiv:1908.10084; model cards for microsoft/deberta-v3-small and sentence-transformers/all-MiniLM-L6-v2; Kaggle U.S. Patent Phrase to Phrase Matching competition. Only the competition's train/test CSVs are used; the later released full rated dataset is excluded.

## Download the result ZIP
The next cell collects small outputs only. Also download this notebook **with outputs** after the saved run finishes; Kaggle does not place its notebook file in /kaggle/working automatically.

In [ ]:
#!/usr/bin/env python3
"""Collect review outputs without datasets, environments or model weights. Standard library only."""
import argparse
from pathlib import Path
import zipfile
import json
import hashlib

SKIP_DIRS={'.venv','venv','.git','__pycache__','.cache','node_modules','site-packages',
           'patent_runtime_wheels','deberta-v3-small','all-MiniLM-L6-v2','data','adapter',
           'trainer','trainer_tmp','patent_weights','reference_evidence'}
JSON_NAMES={'best_config.json','run_summary.json','test_metrics.json','history.json','CHECKS.json'}
SKIP_FILES={'train.csv','test.csv','sample_submission.csv','IndianFoodDatasetCSV.csv',
            'tokenizer.json','config.json','tokenizer_config.json','adapter_config.json'}
SUFFIXES={'.ipynb','.csv','.png','.jpg','.jpeg','.pdf','.log','.txt'}

def eligible(path):
    if path.is_symlink() or not path.is_file():return False
    if any(p in SKIP_DIRS or p.startswith('checkpoint-') for p in path.parts):return False
    if path.name in SKIP_FILES:return False
    if path.suffix.lower()=='.json':return path.name in JSON_NAMES
    if path.suffix.lower() not in SUFFIXES:return False
    if path.suffix.lower()=='.txt' and path.name not in {'notebook_urls.txt','errors.txt','review_notes.txt'}:return False
    return True

def collect(roots,destination):
    destination=destination.resolve();destination.parent.mkdir(parents=True,exist_ok=True)
    count=0;manifest=[]
    with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED) as archive:
        for index,root in enumerate(roots,1):
            root=root.resolve()
            if not root.exists():raise FileNotFoundError(root)
            paths=[root] if root.is_file() else sorted(root.rglob('*'))
            for path in paths:
                if path.resolve()==destination or not eligible(path):continue
                relative=Path(path.name) if root.is_file() else path.relative_to(root)
                arc=Path(f'{index:02d}_{root.stem}')/relative
                archive.write(path,str(arc));count+=1
                manifest.append({'path':str(arc),'bytes':path.stat().st_size,
                                 'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
        archive.writestr('RETURN_MANIFEST.json',json.dumps(manifest,indent=2))
    if count==0:
        destination.unlink();raise ValueError('No review outputs found. Check the input folder.')
    print(f'Saved {count} files: {destination} ({destination.stat().st_size/1e6:.2f} MB)',flush=True)


if Path("/kaggle/working").exists():
    collect([Path("/kaggle/working")],Path("/kaggle/working/A24ARIU0001_PATENT_EXPERIMENTS_RESULTS.zip"))
